<a href="https://colab.research.google.com/github/gibranfp/CursoAprendizajeProfundo/blob/2027-1/notebooks/1d_autodiff_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diferenciación automática
La [diferenciación automática](https://en.wikipedia.org/wiki/Automatic_differentiation) es un método para calcular derivadas de funciones expresadas como programas de forma eficiente.

![Diferenciación automática](https://gowrishankar.info/blog/automatic-differentiation-using-gradient-tapes/auto_diff.png)

Fuente: [Automatic Differentiation in Machine Learning: a Survey, Baydin et. al, 2018](https://arxiv.org/abs/1502.05767).

In [1]:
import numpy as np
import torch as th
from torch import nn

th.manual_seed(42)
np.random.seed(42)

## Clase `Parameter`
La clase [`Parameter`](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) del módulo `nn` de PyTorch define una subclase de `Tensor` que se emplea comúnmente para representar los parámetros que modifican los algoritmos de aprendizaje para generar un modelo. Las instancias de `Parameter` que se definen dentro de una subclase de `Module` del módulo de `nn` se agregan a su lista de parámetros a optimizar.

El constructor de la clase `Parameter` recibe un tensor como argumento con el que se crea la instancia.


In [2]:
print(nn.parameter.Parameter(th.zeros((10,5))))
print(nn.parameter.Parameter(th.rand((10,5))))
print(nn.parameter.Parameter(th.ones((5,5))))
print(nn.parameter.Parameter(th.tensor([[1.0, 2.0], [3.0, 4.0]])))

Parameter containing:
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], requires_grad=True)
Parameter containing:
tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904],
        [0.6009, 0.2566, 0.7936, 0.9408, 0.1332],
        [0.9346, 0.5936, 0.8694, 0.5677, 0.7411],
        [0.4294, 0.8854, 0.5739, 0.2666, 0.6274],
        [0.2696, 0.4414, 0.2969, 0.8317, 0.1053],
        [0.2695, 0.3588, 0.1994, 0.5472, 0.0062],
        [0.9516, 0.0753, 0.8860, 0.5832, 0.3376],
        [0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
        [0.6343, 0.3644, 0.7104, 0.9464, 0.7890],
        [0.2814, 0.7886, 0.5895, 0.7539, 0.1952]], requires_grad=True)
Parameter containing:
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
  

Podemos realizar cualquier operación tensorial con una instancia de `Parameter`, ya sea con otras instancias de `Parameter` o de `Tensor`. El resultado de la operación es una instancia de `Tensor`.

In [3]:
param = nn.parameter.Parameter(th.zeros(10, 5))

print(param.T)
print(param + th.ones_like(param))
print(param * th.zeros_like(param))
print(param @ th.rand((5, 10)))

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], grad_fn=<PermuteBackward0>)
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], grad_fn=<AddBackward0>)
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 

## Diferenciación automática en PyTorch
PyTorch puede diferenciar automáticamente secuencias de operaciones con instancias de `Tensor` o `Parameter`. Para ello se mantiene una gráfica de cómputo, la cual se va generando de manera dinámica conforme se ejecutan operaciones con instancias que tienen la propiedad `requires_grad` en verdadero (solo ten cuidado con las operaciones _in-place_). Por defecto, todas las instancias de `Parameter` tienen esta propiedad en verdadero y las instancias de `Tensor` en falso.

In [4]:
ten = th.zeros((10, 5))

print(f'Parameter: {param.requires_grad}, Tensor: {ten.requires_grad}')

Parameter: True, Tensor: False


Es posible cambiar el valor de esta propiedad en una instancia usando el método `requires_grad_` (_in-place_).

In [5]:
param.requires_grad_(False)
ten.requires_grad_(True)
print(f'Parameter: {param.requires_grad}, Tensor: {ten.requires_grad}')

# lo regresamos a verdadero
param.requires_grad_(True)

Parameter: False, Tensor: True


Parameter containing:
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], requires_grad=True)

Para modificar el contenido de una instancia de `Parameter` es necesario especificar que no se registre la operación usando el ámbito `no_grad`.

In [6]:
with th.no_grad():
  param[0, 0] = 1

print(f'Param={param}')

Param=Parameter containing:
tensor([[1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], requires_grad=True)


En general, existen métodos para instancias tanto de `Parameter` como de `Tensor` que modifican el contenido _in-place_. Los nombres de estos métodos usualmente terminan con un guión bajo. Tal es el caso de `add_` y `mul_`, que suman y multiplican un tensor. Debido a que estas operaciones no deben registrarse cuando las instancias de `Parameter` o `Tensor` tienen `requires_grad = True`, las ejecutamos dentro del ámbito `no_grad`.

In [7]:
with th.no_grad():
  param.add_(th.ones_like(param))
  param.mul_(-th.ones_like(param))
  param.sub_(2 * th.ones_like(param))

print(f'Param={param}')

Param=Parameter containing:
tensor([[-4., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.],
        [-3., -3., -3., -3., -3.]], requires_grad=True)


Para obtener el gradiente de una [función escalar](https://en.wiktionary.org/wiki/scalar_function) respecto a un tensor hoja con `requires_grad=True`, se debe llamar al método `backward` sobre dicha función.

Por ejemplo, considera la siguiente función:

$$
f(x, y) = 4x^2 + 2y^3 + c
$$

![](https://raw.githubusercontent.com/gibranfp/CursoAprendizajeProfundo/refs/heads/2027-1/figs/autodiff.svg)

Evaluando esta función en $x = 2$, $y = 3$ y $c = 1.0$, tenemos

$$
f(2, 3) = 4\cdot (2)^2 + 2 \cdot (3)^3 + 1 = 70
$$

Las derivadas partiales evaluadas en estos puntos estarían dadas por
$$
\begin{align}
\frac{\partial f}{\partial x} = 8x = 8 \cdot (2) = 16\\
\frac{\partial f}{\partial y} = 6y^2 = 6 (3)^2 = 54
\end{align}
$$

In [8]:
x = th.tensor(2.0, requires_grad = True)
y = th.tensor(3.0, requires_grad = True)
c = th.tensor(1.0)

f = 4 * x**2 + 2 * y**3 + c

f.backward()

Cuando se invoca a este método, se calculan los gradientes de la función respecto al tensor correspondiente y se acumulan en el tensor `grad` que es una propiedad de todas las instancia de `Tensor` y `Parameter`.

In [9]:
print(f'f={f}, gx={x.grad}, gy={y.grad}')

f=71.0, gx=16.0, gy=54.0


Además, al construir la gráfica de cómputo de una operación, PyTorch almacena la función de retropropagación correspondiente en la propiedad `grad_fn` del tensor resultante. Para el ejemplo anterior, esta sería:

In [10]:
a = x**3
b = 2 * a

d = y**2
e = 3 * d

g = b + e
f = g + c

print(f'agf={a.grad_fn}')
print(f'bgf={b.grad_fn}')
print(f'dgf={d.grad_fn}')
print(f'egf={e.grad_fn}')
print(f'ggf={g.grad_fn}')
print(f'fgf={f.grad_fn}')

agf=<PowBackward0 object at 0x7e51b054f7f0>
bgf=<MulBackward0 object at 0x7e51b054fc40>
dgf=<PowBackward0 object at 0x7e51b054fc40>
egf=<MulBackward0 object at 0x7e51b054fc40>
ggf=<AddBackward0 object at 0x7e51b054fc40>
fgf=<AddBackward0 object at 0x7e51b054fc40>


Podemos evaluar estas funciones:

In [11]:
a.grad_fn(th.tensor(1.))

tensor(12., grad_fn=<MulBackward0>)

In [12]:
b.grad_fn(th.tensor(1.))

(tensor(2.), None)

In [13]:
a.grad_fn(b.grad_fn(th.tensor(1.))[0])

tensor(24., grad_fn=<MulBackward0>)

In [14]:
print(a.grad_fn(b.grad_fn(g.grad_fn(f.grad_fn(th.tensor(1.))[0])[0])[0]))
print(d.grad_fn(e.grad_fn(g.grad_fn(f.grad_fn(th.tensor(1.))[0])[0])[0]))

tensor(24., grad_fn=<MulBackward0>)
tensor(18., grad_fn=<MulBackward0>)


Cuando se llama al método `backward` se van evaluando las funciones de retropropagación en la gráfica de cómputo desde el último nodo hacia atrás usando la propiedad `next_functions` de `grad_fn` hasta obtener los gradientes de los tensores correspondientes en los nodos hoja. Nota que en algunos casos es necesario mantener los resultados intermedios para poder evaluar esta función y obtener el gradiente correspondiente.

Consideremos ahora la función:

$$
g(\mathbf{m}) =  \sum_{j=1}^d m_j^2
$$

La derivada parcial de esta función respecto a cada elemento de $\mathbf{m}$ estaría dada por

$$
\frac{\partial g(\mathbf{m})}{\partial m_j} = 2\cdot m_j
$$

Definimos esta función, la evalúamos para 100 valores entre -10 y 10 y obtenemos las derivadas parciales respecto a cada uno de los 100 valores (gradiente) usando el método `backward`:

In [15]:
m = th.linspace(start = -10, end = 10, steps = 100, requires_grad = True)
g = (m**2).sum()
g.backward()

print(f'g={g}\n g_grad_func={g.grad_fn}\n m={m}\n gm={m.grad}')

g=3400.67333984375
 g_grad_func=<SumBackward0 object at 0x7e51b0cf8520>
 m=tensor([-10.0000,  -9.7980,  -9.5960,  -9.3939,  -9.1919,  -8.9899,  -8.7879,
         -8.5859,  -8.3838,  -8.1818,  -7.9798,  -7.7778,  -7.5758,  -7.3737,
         -7.1717,  -6.9697,  -6.7677,  -6.5657,  -6.3636,  -6.1616,  -5.9596,
         -5.7576,  -5.5556,  -5.3535,  -5.1515,  -4.9495,  -4.7475,  -4.5455,
         -4.3434,  -4.1414,  -3.9394,  -3.7374,  -3.5354,  -3.3333,  -3.1313,
         -2.9293,  -2.7273,  -2.5253,  -2.3232,  -2.1212,  -1.9192,  -1.7172,
         -1.5152,  -1.3131,  -1.1111,  -0.9091,  -0.7071,  -0.5051,  -0.3030,
         -0.1010,   0.1010,   0.3030,   0.5051,   0.7071,   0.9091,   1.1111,
          1.3131,   1.5152,   1.7172,   1.9192,   2.1212,   2.3232,   2.5253,
          2.7273,   2.9293,   3.1313,   3.3333,   3.5354,   3.7374,   3.9394,
          4.1414,   4.3434,   4.5455,   4.7475,   4.9495,   5.1515,   5.3535,
          5.5556,   5.7576,   5.9596,   6.1616,   6.3636,   6.5657,

Una vez que se llama al método `.backward`, se elimina el grafo de cómputo. Sin embargo, es posible mantenerlo pasando el argumento `retain_graph=True` en la llamada.

In [16]:
g = (m**2).sum()
g.backward(retain_graph=True)
print(f'gm 1={m.grad}')
g.backward()
print(f'gm 2={m.grad}')

gm 1=tensor([-40.0000, -39.1919, -38.3838, -37.5758, -36.7677, -35.9596, -35.1515,
        -34.3434, -33.5354, -32.7273, -31.9192, -31.1111, -30.3030, -29.4949,
        -28.6869, -27.8788, -27.0707, -26.2626, -25.4545, -24.6465, -23.8384,
        -23.0303, -22.2222, -21.4141, -20.6061, -19.7980, -18.9899, -18.1818,
        -17.3737, -16.5657, -15.7576, -14.9495, -14.1414, -13.3333, -12.5253,
        -11.7172, -10.9091, -10.1010,  -9.2929,  -8.4848,  -7.6768,  -6.8687,
         -6.0606,  -5.2525,  -4.4444,  -3.6364,  -2.8283,  -2.0202,  -1.2121,
         -0.4040,   0.4040,   1.2121,   2.0202,   2.8283,   3.6364,   4.4444,
          5.2525,   6.0606,   6.8687,   7.6768,   8.4848,   9.2929,  10.1010,
         10.9091,  11.7172,  12.5253,  13.3333,  14.1414,  14.9495,  15.7576,
         16.5657,  17.3737,  18.1818,  18.9899,  19.7980,  20.6061,  21.4141,
         22.2222,  23.0303,  23.8384,  24.6465,  25.4545,  26.2626,  27.0707,
         27.8788,  28.6869,  29.4949,  30.3030,  31.1111,  

Múltiples llamadas a la función y al método `backward` acumulan los gradientes en `.grad`. Por lo mismo, en muchas ocasiones es necesarios ponerlos a 0 con el método (_in-pace_) `zero_`.

In [17]:
# no requiere estar en el ámbito no_grad porque el tensor
# grad tiene requires_grad=False
m.grad.zero_()
print(f'gm={m.grad}')

gm=tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])


También podemos diferenciar automáticamente respecto a más de un tensor.

In [18]:
l = th.rand_like(m, requires_grad=True)
h = (m**2 + l**3).sum()
h.backward()

print(f'h={h}, h_grad_func={h.grad_fn}, gm={m.grad}, gl={l.grad}')

h=3421.044677734375, h_grad_func=<SumBackward0 object at 0x7e51b0cf8520>, gm=tensor([-20.0000, -19.5960, -19.1919, -18.7879, -18.3838, -17.9798, -17.5758,
        -17.1717, -16.7677, -16.3636, -15.9596, -15.5556, -15.1515, -14.7475,
        -14.3434, -13.9394, -13.5354, -13.1313, -12.7273, -12.3232, -11.9192,
        -11.5152, -11.1111, -10.7071, -10.3030,  -9.8990,  -9.4949,  -9.0909,
         -8.6869,  -8.2828,  -7.8788,  -7.4747,  -7.0707,  -6.6667,  -6.2626,
         -5.8586,  -5.4545,  -5.0505,  -4.6465,  -4.2424,  -3.8384,  -3.4343,
         -3.0303,  -2.6263,  -2.2222,  -1.8182,  -1.4141,  -1.0101,  -0.6061,
         -0.2020,   0.2020,   0.6061,   1.0101,   1.4141,   1.8182,   2.2222,
          2.6263,   3.0303,   3.4343,   3.8384,   4.2424,   4.6465,   5.0505,
          5.4545,   5.8586,   6.2626,   6.6667,   7.0707,   7.4747,   7.8788,
          8.2828,   8.6869,   9.0909,   9.4949,   9.8990,  10.3030,  10.7071,
         11.1111,  11.5152,  11.9192,  12.3232,  12.7273,  13.131

Por otro lado, es posible obtener derivadas de orden mayor, pero esto se logra usando la función `grad` del módulo `autograd` en lugar del método `backward`.

In [19]:
z = th.rand(100, requires_grad=True)
f = (z**3).sum()
df = th.autograd.grad(f, z, create_graph=True)[0]
d2f = th.autograd.grad(df.sum(), z)[0]

print(f'f={f}, grad_func={f.grad_fn}')
print(f'df={df}, grad_func={df.grad_fn}')
print(f'z={z}, grad_func={z.grad_fn}')
print(f'd2f={d2f}, grad_func={d2f.grad_fn}')

f=20.557010650634766, grad_func=<SumBackward0 object at 0x7e51b054f940>
df=tensor([6.3126e-03, 2.9859e-01, 2.5451e+00, 1.4481e+00, 6.7725e-01, 1.1826e-01,
        1.1302e-01, 8.1484e-03, 3.4075e-01, 1.3421e+00, 2.0114e+00, 1.6024e+00,
        1.0102e-02, 1.1918e-01, 5.3195e-01, 2.9028e+00, 9.8268e-01, 4.1184e-01,
        1.4989e+00, 2.8748e-01, 9.3321e-02, 2.2444e+00, 2.2301e-01, 4.7944e-01,
        2.0247e-05, 2.0898e+00, 2.3170e+00, 1.3963e+00, 6.8732e-02, 1.2792e-04,
        2.6458e-02, 2.2856e+00, 1.6430e+00, 2.5434e+00, 1.7416e+00, 1.1777e+00,
        7.3538e-01, 4.3018e-02, 1.5386e-02, 3.1348e-03, 1.4897e+00, 1.9434e-01,
        4.7850e-01, 1.3515e-01, 5.0157e-01, 6.5785e-02, 9.0091e-02, 1.3301e+00,
        3.7045e-01, 1.9618e+00, 3.4598e-01, 5.3240e-02, 5.0869e-01, 1.9911e-01,
        3.6129e-01, 1.7283e-03, 1.8240e+00, 6.9219e-02, 1.6934e+00, 1.5851e+00,
        2.2045e+00, 4.0699e-02, 2.2167e+00, 2.0849e-01, 1.4099e+00, 2.8201e+00,
        5.5337e-01, 7.3844e-01, 4.4441e-01, 2

## Neurona artificial
La salida de una neurona artificial se obtiene multiplicando la transpuesta del vector de pesos $\mathbf{w}\in \mathbb{R}^{d\times 1}$ por el vector de entrada $\mathbf{x} \in \mathbb{R}^{1\times d}$, sumando al final el valor del sesgo $b\in \mathbb{R^{1\times 1}}$ y evaluando el resultado con la función de activación $\phi$, esto es

$$
a = \phi\left(\mathbf{x} \mathbf{w} + b\right)
$$

![Diagrama general de la neurona artificial](http://turing.iimas.unam.mx/~gibranfp/cursos/neurona.svg)

Una neurona artificial con función de activación logística o sigmoide entrenada minimizando la [entropía cruzada binaria](https://en.wikipedia.org/wiki/Cross_entropy) se corresponde con una [regresión logística](https://en.wikipedia.org/wiki/Logistic_regression). La [función sigmoide](https://en.wikipedia.org/wiki/Sigmoid_function) o logística está dada por

$$
\sigma(z) = \frac{1}{1 + e^{-z}},
$$

Por lo tanto, la salida de la neurona sigmoide o logística sería

$$
\hat{y} = \sigma(\mathbf{x} \mathbf{w} + b)
$$



In [20]:
def neurona_sigmoide(X, w, b):
  return th.sigmoid(X @ w + b)

Por su parte, la entropía cruzada binaria está dada por
$$
ECB(\mathbf{\hat{y}}, \mathbf{y}) = -\frac{1}{n}\sum_{i=1}^n \left[y^{(i)} \log{(\hat{y}^{(i)})} + (1 - y^{(i)}) \log{(1 - \hat{y}^{(i)})}\right]
$$

In [21]:
def ecb( y_hat, y):
  perdida_unos = th.log(y_hat[y == 1]).sum()
  perdida_ceros = th.log(1 - y_hat[y == 0]).sum()
  return -(perdida_unos + perdida_ceros) / y.shape[0]

Generamos un conjunto de ejemplos sintéticos (aleatorios).

In [22]:
n = 100
d = 10
X = th.normal(size = (n, d), mean = 0, std = 1)
y = th.randint(low = 0, high = 2, size = (n, 1))

print(X, y)

tensor([[-5.5719e-01, -9.6835e-01,  8.7128e-01, -9.5641e-02,  4.0380e-01,
         -7.1398e-01,  8.3373e-01, -9.5855e-01,  1.0682e+00, -2.5272e-01],
        [-1.8815e-01, -7.7115e-01,  1.7989e-01, -2.1268e+00, -1.3408e-01,
         -1.0408e+00,  7.6942e-01,  2.5574e+00,  5.7161e-01,  1.3596e+00],
        [ 4.3344e-01, -7.1719e-01,  1.0554e+00, -1.4534e+00,  1.7361e+00,
          1.8350e+00,  8.8002e-01,  5.6080e-02,  3.7818e-01,  7.0511e-01],
        [-1.7237e+00, -8.4348e-01, -4.8619e-01, -3.3600e-01,  3.6716e-02,
          4.9340e-01,  8.8538e-01,  1.8244e-01,  7.8638e-01, -5.7920e-02],
        [ 1.3637e-01,  3.0880e-01,  1.6617e+00,  1.7512e-01,  6.0841e-01,
          1.6309e+00, -8.4723e-02,  1.0844e+00,  1.9537e-01, -1.3350e+00],
        [ 3.9451e-01,  1.7060e+00, -7.9394e-01,  3.7523e-01,  8.7910e-02,
         -1.2415e+00, -5.6626e-01,  3.9892e-01,  1.3695e+00, -2.5189e-01],
        [ 1.9003e+00,  1.6951e+00,  2.8090e-02, -1.7537e-01,  4.0854e-01,
         -1.2609e+00,  9.1652e-0

 Creamos tensores para los pesos inicializados con valores aleatorios muestreados de una normal ($\mu = 0$ y $\sigma = 1$) y el sesgo inicializado con cero.

In [23]:
w = nn.parameter.Parameter(th.randn((d, 1)))
b = nn.parameter.Parameter(th.zeros((1, 1)))

print(w, b)

Parameter containing:
tensor([[-1.5897],
        [-0.0405],
        [ 1.9010],
        [-0.6620],
        [ 0.2589],
        [-1.0627],
        [-1.4913],
        [ 0.1673],
        [ 0.7528],
        [ 0.6113]], requires_grad=True) Parameter containing:
tensor([[0.]], requires_grad=True)


Podemos obtener los gradientes de $\mathbf{w}$ y $b$ respecto a la [entropía cruzada binaria](https://en.wikipedia.org/wiki/Cross_entropy) usando diferenciación automática.

![](https://pytorch.org/tutorials/_images/comp-graph.png)

Fuente: Tutorial [Automatic Differentiation with `torch.autograd`](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

Nota que en esta gráfica de cómputo no se observa la función sigmoide. Esto se debe a que en PyTorch hay una versión de la entropía cruzada binaria que recibe los _logits_ como entrada.



In [24]:
y_hat = neurona_sigmoide(X, w, b)
fp = ecb(y_hat, y)
fp.backward()

print(w.grad, b.grad)

tensor([[-0.1555],
        [ 0.1013],
        [ 0.1673],
        [-0.0567],
        [ 0.0159],
        [-0.1490],
        [-0.1399],
        [ 0.0444],
        [ 0.0939],
        [ 0.1275]]) tensor([[-0.0712]])


## Ejercicio
Genera un conjunto de datos sintético, programa la propagación hacia adelante y calcula los gradientes de la función de pérdida del error cuadrático medio respecto a los pesos y sesgos para $K$ regresiones lineales.